# Module 6 — Notebook 4: Circuit QED, Transmon Qubits & Dispersive Readout

[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)
[![NumPy](https://img.shields.io/badge/numpy-2.0+-green.svg)](https://numpy.org/)
[![SciPy](https://img.shields.io/badge/scipy-1.12+-teal.svg)](https://scipy.org/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Publication Quality](https://img.shields.io/badge/Visualizations-Publication%20Quality-indigo.svg)]()

---

### Executive Overview & Abstract

Circuit Quantum Electrodynamics (**circuit QED / cQED**) is the leading hardware paradigm for scalable solid-state quantum computation. By adapting the principles of Cavity QED to microwave-frequency superconducting circuits, microscopic atoms are replaced by macroscopic superconducting **artificial atoms** (such as the Cooper Pair Box and the Transmon), and Fabry-Pérot optical cavities are replaced by superconducting coplanar waveguide (CPW) transmission line resonators or 3D microwave cavities.

Early superconducting qubits were severely limited by $1/f$ charge noise. In 2007, **Jens Koch et al.** introduced the **Transmon** (transmission-line shunted plasma oscillation qubit), operating in the regime where the Josephson coupling energy dominates the electrostatic charging energy:
$$\frac{E_J}{E_C} \gg 1 \quad \left( \text{typically } \frac{E_J}{E_C} \approx 20 - 80 \right)$$
In this regime:
1. **Exponential Charge Noise Immunity**: The charge dispersion $\epsilon_k = \max_{n_g} E_k - \min_{n_g} E_k$ decreases exponentially:
   $$\epsilon_k \propto \exp\left( -\sqrt{8 E_J / E_C} \right)$$
   suppressing dephasing by over four orders of magnitude and rendering the energy bands completely flat with respect to offset gate charge $n_g$.
2. **Weak Duffing Anharmonicity**: The transmon retains a negative anharmonicity $\alpha = \omega_{12} - \omega_{01} \approx -E_C$, which scales only algebraically as a power law $\mathcal{O}((E_C/E_J)^{1/4})$, ensuring sufficient level non-linearity to selectively isolate a computational two-level subspace $\{|0\rangle, |1\rangle\}$.
3. **Dispersive Quantum Non-Demolition (QND) Readout**: In the dispersive regime where the qubit-cavity detuning exceeds the coupling rate ($|\Delta| = |\omega_{01} - \omega_r| \gg g$), the effective Hamiltonian is:
   $$\hat{H}_{\mathrm{disp}} \approx \hbar \omega_r \hat{a}^\dagger\hat{a} + \frac{\hbar \tilde{\omega}_q}{2} \hat{\sigma}_z + \hbar \chi \hat{a}^\dagger\hat{a} \hat{\sigma}_z$$
   Because $[\hat{H}_{\mathrm{disp}}, \hat{\sigma}_z] = 0$, measuring the transmitted microwave probe through the cavity yields the qubit state without inducing unwanted state transitions (QND measurement).
4. **Photon-Number-Resolved Spectroscopy**: In the strong dispersive regime ($\chi > \kappa, \gamma$), the qubit resonance frequency splits into a discrete comb $\omega_q(n) = \omega_q - 2n\chi$, allowing direct photon number counting and mapping of non-classical cavity states.

---

### Pedagogical & Computational Roadmap

1. **Theoretical Foundations**: The Cooper Pair Box Hamiltonian in charge and phase representations, Mathieu equation solutions, transmon asymptotic limits, and the Schrieffer-Wolff dispersive transformation.
2. **Computational Engine (`CircuitQEDEngine`)**:
   - Exact diagonalization of the Cooper Pair Box Hamiltonian in the truncated Cooper pair charge basis $\{|n\rangle\}_{n=-N_{\max}}^{+N_{\max}}$.
   - Extraction of transition frequencies $\omega_{01}$, anharmonicity $\alpha$, and gate charge dispersion $\epsilon_{01}$.
   - Dispersive cavity transmission $S_{21}(\omega)$ and IQ phasor separation.
   - Qubit spectroscopy with photon-number-resolved splitting for Fock and coherent microwave states.
3. **Publication-Grade 4-Panel Visualization**:
   - *Panel (a)*: Transmon Energy Spectrum vs. Gate Charge $n_g$: Comparing CPB ($E_J/E_C = 1$, high dispersion) vs. Transmon ($E_J/E_C = 50$, flat bands).
   - *Panel (b)*: Exponential Suppression of Charge Dispersion: $\epsilon_{01}$ vs. $E_J/E_C$ validating $\epsilon \propto \exp(-\sqrt{8E_J/E_C})$.
   - *Panel (c)*: Dispersive QND Readout: Cavity transmission spectrum $|S_{21}(\omega)|^2$ showing the $2\chi$ frequency shift between $|g\rangle$ and $|e\rangle$.
   - *Panel (d)*: Photon-Number-Resolved Spectroscopy: Qubit absorption spectrum $S_q(\omega)$ resolving discrete peaks for Fock states $n = 0, 1, 2, 3, 4$ and coherent state Poissonian envelopes.
4. **Interactive Exploration Suite**: Real-time tuning of $E_J/E_C$, dispersive shift $\chi$, cavity linewidth $\kappa$, and mean photon number $\bar{n}$.
5. **Rigorous Automated Benchmark Suite**: Assert transmon asymptotic frequency $\omega_{01} \approx \sqrt{8E_J E_C} - E_C$, negative anharmonicity $\alpha \approx -E_C$, exponential dispersion suppression, and exact $2\chi$ spectroscopic peak separation.


In [ ]:
# Cell 2: Setup, Imports, and Publication Styling Configuration
import sys
import os
import math
import numpy as np
import scipy.linalg as la
import scipy.special as sp
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Optional, Union

# Set random seed for reproducibility
np.random.seed(42)

# Publication-grade plotting parameters
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.titlesize": 16,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "lines.linewidth": 2.0,
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
})

print(f"NumPy version: {np.__version__}")
print("Circuit QED & Transmon environment initialized successfully.")


---

## Theoretical Deep Dive: Superconducting Qubits & Dispersive cQED

### 1. The Cooper Pair Box Hamiltonian

A superconducting island connected to a reservoir via a Josephson junction is governed by the competition between electrostatic charging energy $E_C = \frac{e^2}{2 C_\Sigma}$ and Josephson tunneling energy $E_J = \frac{I_c \Phi_0}{2\pi}$.
In the charge basis $|n\rangle$ (where $n$ is the number of transferred Cooper pairs):
$$\hat{H}_{\mathrm{CPB}} = 4 E_C \left( \hat{n} - n_g \right)^2 - E_J \cos\hat{\phi} = 4 E_C \sum_{n} (n - n_g)^2 |n\rangle\langle n| - \frac{E_J}{2} \sum_{n} \left( |n\rangle\langle n+1| + |n+1\rangle\langle n| \right)$$
where $n_g = \frac{C_g V_g}{2e}$ is the continuous dimensionless offset charge induced by nearby electrostatic gate voltage or background quasiparticle charge fluctuations.

In the phase representation ($\hat{\phi} \in [0, 2\pi)$):
$$\hat{H}_{\mathrm{CPB}} = 4 E_C \left( -i\frac{\partial}{\partial \phi} - n_g \right)^2 - E_J \cos\phi$$
This is the celebrated **Mathieu equation**, whose eigenenergies $E_k(n_g)$ are periodic in $n_g$ with period 1.

---

### 2. The Transmon Regime: Asymptotic Flattening of Charge Bands

In the traditional Cooper Pair Box ($E_J / E_C \sim 1$), the energy levels $E_k(n_g)$ exhibit large variations with gate charge ($\Delta E \sim E_C$). Low-frequency $1/f$ charge noise causes rapid dephasing ($T_2^* \sim 1 - 10\ \mathrm{ns}$).
By shunting the junction with a large interdigital or coplanar capacitor ($C_\Sigma \gg C_J$), the transmon operates at:
$$\frac{E_J}{E_C} \gg 1 \quad (20 \le E_J/E_C \le 80)$$

In this deep Josephson potential well, the phase fluctuations are small ($\langle\phi^2\rangle \ll 1$). Expanding the cosine potential:
$$\cos\phi \approx 1 - \frac{\phi^2}{2} + \frac{\phi^4}{24} + \dots$$
The system behaves as a **weakly non-linear Duffing oscillator**:
1. **Transition Frequency**:
   $$\hbar \omega_{01} = E_1 - E_0 \approx \sqrt{8 E_J E_C} - E_C$$
2. **Anharmonicity (Negative Duffing Non-Linearity)**:
   $$\alpha = \omega_{12} - \omega_{01} \approx -E_C$$
   Because $\alpha$ scales only as $\mathcal{O}((E_C/E_J)^{1/4})$, it remains comfortably large ($\approx -200$ to $-350\ \mathrm{MHz}$), easily allowing selective $20\ \mathrm{ns}$ microwave pulse driving.
3. **Exponential Suppression of Charge Dispersion**:
   WKB tunneling between adjacent phase minima yields an exponential suppression of charge sensitivity:
   $$\epsilon_k = E_k(n_g = 0.5) - E_k(n_g = 0) \propto (-1)^k \frac{2^{4k+5}}{k!} \sqrt{\frac{2}{\pi}} \left( \frac{E_J}{2E_C} \right)^{\frac{k}{2} + \frac{3}{4}} \exp\left( -\sqrt{\frac{8E_J}{E_C}} \right)$$
   For $E_J / E_C = 50$, $\epsilon_1 \approx 10\ \mathrm{kHz} \ll E_C$, making the qubit completely immune to charge noise and boosting $T_2^*$ to hundreds of microseconds!

---

### 3. The Dispersive Jaynes-Cummings Regime & QND Readout

When the transmon is capacitively coupled to a transmission line resonator with coupling strength $g$, the multi-level Jaynes-Cummings Hamiltonian is:
$$\hat{H} = \hbar \omega_r \hat{a}^\dagger\hat{a} + \sum_k E_k |k\rangle\langle k| + \hbar \sum_{j, k} g_{jk} \left( |j\rangle\langle k| \hat{a} + |k\rangle\langle j| \hat{a}^\dagger \right)$$
In the **dispersive regime**, the detuning $|\Delta| = |\omega_{01} - \omega_r|$ is much larger than the coupling rate ($|\Delta| \gg g$). Performing a second-order Schrieffer-Wolff transformation truncating to the $\{|0\rangle, |1\rangle\} \equiv \{|g\rangle, |e\rangle\}$ computational subspace yields:
$$\hat{H}_{\mathrm{disp}} \approx \hbar \left( \omega_r + \chi \hat{\sigma}_z \right) \hat{a}^\dagger\hat{a} + \frac{\hbar \tilde{\omega}_q}{2} \hat{\sigma}_z$$
where the **dispersive shift** $\chi$ incorporates both the $|0\rangle \leftrightarrow |1\rangle$ and $|1\rangle \leftrightarrow |2\rangle$ transitions:
$$\chi = \chi_{01} - \frac{\chi_{12}}{2} \approx -\frac{g^2}{\Delta} \frac{E_C}{\Delta - E_C}$$

#### Quantum Non-Demolition (QND) Property:
Because $[\hat{H}_{\mathrm{disp}}, \hat{\sigma}_z] = 0$, measuring the cavity field does not induce transitions between $|g\rangle$ and $|e\rangle$.
- If the qubit is in ground state $|g\rangle$ ($\sigma_z = -1$), the cavity frequency is $\omega_r - \chi$.
- If the qubit is in excited state $|e\rangle$ ($\sigma_z = +1$), the cavity frequency is $\omega_r + \chi$.
Sending a microwave probe tone at $\omega_r$ shifts the transmission amplitude $S_{21}(\omega)$ and output IQ phase by $\Delta\phi \approx 2\arctan(2\chi / \kappa)$, providing fast single-shot projective readout!

---

### 4. Photon-Number-Resolved Spectroscopy

Conversely, rewriting the interaction as $\hbar (\tilde{\omega}_q + 2\chi \hat{a}^\dagger\hat{a}) \frac{\hat{\sigma}_z}{2}$ shows that **the qubit transition frequency is dressed by the number of intracavity photons**:
$$\omega_q(n) = \omega_{01} - 2n\chi$$
In the strong dispersive regime where the dispersive shift exceeds the qubit and cavity linewidths ($\chi > \kappa, \gamma$):
- Each photon added to the cavity shifts the qubit resonance by a distinct step $2\chi$.
- Interrogating the qubit with a weak spectroscopy tone resolves discrete, well-separated absorption peaks corresponding to $n = 0, 1, 2, 3, \dots$ photons.
- The height of each peak directly reflects the photon number distribution $P(n) = |\langle n|\psi\rangle|^2$, providing a direct non-destructive quantum number counter!


In [ ]:
# Cell 4: The Circuit QED & Transmon Computational Engine
class CircuitQEDEngine:
    '''
    Publication-grade computational engine for Circuit QED,
    Cooper Pair Box exact charge basis diagonalization, Transmon physics,
    dispersive QND cavity readout, and photon-number-resolved spectroscopy.
    '''

    @staticmethod
    def diagonalize_cooper_pair_box(
        EJ: float, 
        EC: float, 
        ng: float = 0.0, 
        n_cutoff: int = 15
    ) -> Tuple[np.ndarray, np.ndarray]:
        '''
        Diagonalizes the Cooper Pair Box Hamiltonian in the truncated charge basis
        {|n> : n in [-n_cutoff, n_cutoff]}.
        H = 4 * EC * (n - ng)^2 - 0.5 * EJ * (|n><n+1| + |n+1><n|).
        Returns: (eigenvalues, eigenvectors).
        '''
        n_basis = np.arange(-n_cutoff, n_cutoff + 1)
        dim = len(n_basis)
        H = np.zeros((dim, dim), dtype=float)
        
        # Electrostatic charging diagonal
        diag_elements = 4.0 * EC * (n_basis - ng)**2
        np.fill_diagonal(H, diag_elements)
        
        # Josephson tunneling off-diagonal
        for i in range(dim - 1):
            H[i, i + 1] = -0.5 * EJ
            H[i + 1, i] = -0.5 * EJ
            
        evals, evecs = la.eigh(H)
        return evals, evecs

    @classmethod
    def charge_dispersion(
        cls, 
        EJ: float, 
        EC: float, 
        n_cutoff: int = 15
    ) -> Tuple[float, float, float]:
        '''
        Computes the peak-to-peak gate charge dispersion epsilon = max(E) - min(E)
        for ground state, first excited state, and 01 transition frequency.
        '''
        evals_0, _ = cls.diagonalize_cooper_pair_box(EJ, EC, ng=0.0, n_cutoff=n_cutoff)
        evals_half, _ = cls.diagonalize_cooper_pair_box(EJ, EC, ng=0.5, n_cutoff=n_cutoff)
        
        eps_0 = float(np.abs(evals_0[0] - evals_half[0]))
        eps_1 = float(np.abs(evals_0[1] - evals_half[1]))
        
        omega_01_0 = evals_0[1] - evals_0[0]
        omega_01_half = evals_half[1] - evals_half[0]
        eps_01 = float(np.abs(omega_01_0 - omega_01_half))
        
        return eps_0, eps_1, eps_01

    @staticmethod
    def transmon_asymptotic_formulas(EJ: float, EC: float) -> Tuple[float, float]:
        '''
        Computes transmon transition frequency omega_01 and anharmonicity alpha
        using the standard asymptotic expansion for EJ/EC >> 1:
        omega_01 ~ sqrt(8 * EJ * EC) - EC, alpha ~ -EC.
        '''
        omega_01 = np.sqrt(8.0 * EJ * EC) - EC
        anharmonicity = -EC
        return float(omega_01), float(anharmonicity)

    @staticmethod
    def dispersive_cavity_transmission(
        omega_scan: np.ndarray, 
        omega_r: float, 
        chi: float, 
        kappa: float, 
        qubit_state: str = "g"
    ) -> Tuple[np.ndarray, np.ndarray]:
        '''
        Computes the complex transmission coefficient S_21 and transmittance |S_21|^2
        for cavity readout when qubit is in ground state 'g' (omega_r - chi)
        or excited state 'e' (omega_r + chi).
        '''
        # Resonance frequency shifted by dispersive interaction
        shift = -chi if qubit_state == "g" else +chi
        omega_eff = omega_r + shift
        
        # S_21 = 1 - (kappa / 2) / (i(omega - omega_eff) + kappa / 2)
        denom = 1.0j * (omega_scan - omega_eff) + (kappa / 2.0)
        S21 = 1.0 - (kappa / 2.0) / denom
        
        transmittance = np.abs(S21)**2
        return S21, transmittance

    @staticmethod
    def photon_number_resolved_spectrum(
        omega_scan: np.ndarray, 
        omega_q: float, 
        chi: float, 
        gamma_q: float, 
        photon_distribution: np.ndarray
    ) -> np.ndarray:
        '''
        Computes the qubit absorption spectrum S_q(omega) with photon-number-resolved peaks.
        Each Fock state |n> with probability P(n) produces a peak at omega_q - 2*n*chi.
        '''
        spectrum = np.zeros_like(omega_scan, dtype=float)
        
        for n, p_n in enumerate(photon_distribution):
            if p_n > 1e-6:
                center = omega_q - 2.0 * n * chi
                # Lorentzian absorption line
                lorentz = (gamma_q / 2.0) / ((omega_scan - center)**2 + (gamma_q / 2.0)**2)
                spectrum += p_n * lorentz
                
        return spectrum

print("CircuitQEDEngine loaded successfully.")


In [ ]:
# Cell 5: Publication-Quality 4-Panel Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Circuit QED: Transmon Qubits, Dispersive Readout & Spectroscopy", fontsize=16, y=0.98)

# -----------------------------------------------------------------
# Panel (a): Transmon Energy Bands vs Offset Charge ng
# -----------------------------------------------------------------
ax_a = axes[0, 0]
ng_scan = np.linspace(-1.0, 1.0, 100)
EC_fixed = 0.25 # GHz

# Case 1: CPB regime EJ/EC = 1
EJ_cpb = 1.0 * EC_fixed
e0_cpb, e1_cpb, e2_cpb = [], [], []
for ng in ng_scan:
    ev, _ = CircuitQEDEngine.diagonalize_cooper_pair_box(EJ_cpb, EC_fixed, ng=ng, n_cutoff=10)
    e0_cpb.append(ev[0])
    e1_cpb.append(ev[1])
    e2_cpb.append(ev[2])

# Case 2: Transmon regime EJ/EC = 50
EJ_trans = 50.0 * EC_fixed
e0_trans, e1_trans, e2_trans = [], [], []
for ng in ng_scan:
    ev, _ = CircuitQEDEngine.diagonalize_cooper_pair_box(EJ_trans, EC_fixed, ng=ng, n_cutoff=15)
    e0_trans.append(ev[0])
    e1_trans.append(ev[1])
    e2_trans.append(ev[2])

# Plot CPB bands (dashed)
ax_a.plot(ng_scan, np.array(e0_cpb) - e0_cpb[0], ":", color="#7f7f7f", lw=1.8, label=r"CPB ($E_J/E_C=1$, High Dispersion)")
ax_a.plot(ng_scan, np.array(e1_cpb) - e0_cpb[0], ":", color="#7f7f7f", lw=1.8)
ax_a.plot(ng_scan, np.array(e2_cpb) - e0_cpb[0], ":", color="#7f7f7f", lw=1.8)

# Plot Transmon bands (solid, offset for visual separation)
ax_a.plot(ng_scan, np.array(e0_trans) - e0_trans[0], color="#1f77b4", lw=2.5, label=r"Transmon $E_0$ ($E_J/E_C=50$)")
ax_a.plot(ng_scan, np.array(e1_trans) - e0_trans[0], color="#2ca02c", lw=2.5, label=r"Transmon $E_1$ (Flat Band)")
ax_a.plot(ng_scan, np.array(e2_trans) - e0_trans[0], color="#d62728", lw=2.5, label=r"Transmon $E_2$ (Flat Band)")

ax_a.set_title(r"(a) Energy Bands: CPB vs Transmon Regime", pad=10)
ax_a.set_xlabel(r"Offset Gate Charge $n_g = C_g V_g / 2e$")
ax_a.set_ylabel(r"Energy $(E_k - E_0)$ (GHz)")
ax_a.set_xlim(-1.0, 1.0)
ax_a.set_ylim(-0.5, 11.0)
ax_a.legend(loc="upper right", framealpha=0.9)

# -----------------------------------------------------------------
# Panel (b): Exponential Suppression of Charge Dispersion
# -----------------------------------------------------------------
ax_b = axes[0, 1]
ej_ec_ratios = np.linspace(1.0, 60.0, 60)
disp_01_vals = []
disp_asympt_vals = []

for ratio in ej_ec_ratios:
    EJ_val = ratio * EC_fixed
    _, _, eps_01 = CircuitQEDEngine.charge_dispersion(EJ_val, EC_fixed, n_cutoff=15)
    disp_01_vals.append(eps_01)
    
    # Asymptotic formula epsilon_1
    asympt = EC_fixed * 2.0**9 * np.sqrt(2.0 / np.pi) * (ratio / 2.0)**(1.25) * np.exp(-np.sqrt(8.0 * ratio))
    disp_asympt_vals.append(asympt)

ax_b.semilogy(ej_ec_ratios, disp_01_vals, "o", color="#1f77b4", markersize=5, label=r"Numerical Charge Dispersion $\epsilon_{01}$")
ax_b.semilogy(ej_ec_ratios, disp_asympt_vals, "--", color="#d62728", lw=2.0, label=r"Asymptotic WKB $\propto \exp(-\sqrt{8E_J/E_C})$")
ax_b.axhline(1e-4, color="green", linestyle=":", lw=1.8, label=r"Charge-Noise-Free Threshold ($100\ \mathrm{kHz}$)")

ax_b.set_title(r"(b) Exponential Suppression of Charge Dispersion", pad=10)
ax_b.set_xlabel(r"Ratio $E_J / E_C$")
ax_b.set_ylabel(r"Charge Dispersion $\epsilon_{01}$ (GHz)")
ax_b.set_xlim(1.0, 60.0)
ax_b.set_ylim(1e-7, 1e1)
ax_b.legend(loc="upper right", framealpha=0.9)

# -----------------------------------------------------------------
# Panel (c): Dispersive QND Readout Cavity Transmission S21
# -----------------------------------------------------------------
ax_c = axes[1, 0]
omega_r_res = 7.0 # 7 GHz cavity
chi_disp = 0.005  # 5 MHz dispersive shift
kappa_cav = 0.003 # 3 MHz cavity linewidth
f_scan = np.linspace(6.985, 7.015, 500)

_, T_g = CircuitQEDEngine.dispersive_cavity_transmission(f_scan, omega_r_res, chi_disp, kappa_cav, qubit_state="g")
_, T_e = CircuitQEDEngine.dispersive_cavity_transmission(f_scan, omega_r_res, chi_disp, kappa_cav, qubit_state="e")

ax_c.plot(f_scan * 1e3, T_g, color="#1f77b4", lw=2.5, label=r"Qubit in $|g\rangle$ ($\omega_r - \chi$)")
ax_c.plot(f_scan * 1e3, T_e, color="#d62728", lw=2.5, label=r"Qubit in $|e\rangle$ ($\omega_r + \chi$)")

# Annotate 2*chi split
ax_c.annotate(
    "", xy=((omega_r_res + chi_disp)*1e3, 0.4), xytext=((omega_r_res - chi_disp)*1e3, 0.4),
    arrowprops=dict(arrowstyle="<->", color="black", lw=1.8)
)
ax_c.text(omega_r_res*1e3, 0.45, rf"$2\chi = {2*chi_disp*1e3:.0f}\ \mathrm{{MHz}}$", ha="center", fontweight="bold")

ax_c.set_title(r"(c) Dispersive Cavity Transmission $|S_{21}|^2$ (QND Readout)", pad=10)
ax_c.set_xlabel(r"Probe Frequency (MHz)")
ax_c.set_ylabel(r"Transmittance $|S_{21}|^2$")
ax_c.set_xlim(6.985 * 1e3, 7.015 * 1e3)
ax_c.set_ylim(-0.05, 1.05)
ax_c.legend(loc="upper left", framealpha=0.9)

# -----------------------------------------------------------------
# Panel (d): Photon-Number-Resolved Spectroscopy
# -----------------------------------------------------------------
ax_d = axes[1, 1]
omega_q_bare = 5000.0 # 5 GHz in MHz
chi_spec = 4.0        # 4 MHz dispersive shift
gamma_qubit = 0.8     # 800 kHz qubit linewidth
spec_scan = np.linspace(omega_q_bare - 35.0, omega_q_bare + 5.0, 600)

# Coherent state distribution with mean n_bar = 2.0
alpha_mean = np.sqrt(2.0)
n_photons = 6
p_poisson = [np.exp(-abs(alpha_mean)**2) * (abs(alpha_mean)**(2*n)) / sp.factorial(n) for n in range(n_photons)]

spec_coherent = CircuitQEDEngine.photon_number_resolved_spectrum(spec_scan, omega_q_bare, chi_spec, gamma_qubit, np.array(p_poisson))
ax_d.plot(spec_scan, spec_coherent, color="#9467bd", lw=2.2, label=r"Coherent State ($\bar{n} = 2.0$ Poissonian)")

# Mark individual Fock peaks
for n_idx in range(5):
    peak_loc = omega_q_bare - 2.0 * n_idx * chi_spec
    ax_d.axvline(peak_loc, color="gray", linestyle=":", alpha=0.6)
    ax_d.text(peak_loc, 0.48 - n_idx*0.03, f"$n={n_idx}$", ha="center", fontsize=9, fontweight="bold")

ax_d.set_title(r"(d) Photon-Number-Resolved Qubit Spectroscopy", pad=10)
ax_d.set_xlabel(r"Qubit Drive Frequency (MHz)")
ax_d.set_ylabel(r"Qubit Absorption Signal $S_q(\omega)$")
ax_d.set_xlim(omega_q_bare - 35.0, omega_q_bare + 5.0)
ax_d.set_ylim(-0.05, 0.55)
ax_d.legend(loc="upper left", framealpha=0.9)

plt.tight_layout()
plt.show()


---

## Physical Interpretation & Detailed Mathematical Insights

### 1. The Transmon Miracle: Flat Bands vs. Non-Zero Anharmonicity

Panels (a) and (b) illustrate the central design breakthrough of the transmon qubit:
- In the Cooper Pair Box ($E_J / E_C = 1$), the energy bands oscillate strongly with offset charge $n_g$. Because stray environmental electrons cause $n_g$ to fluctuate randomly, the qubit dephases within nanoseconds ($T_2^* \sim 10\ \mathrm{ns}$).
- As $E_J / E_C$ increases, the wavefunctions become tightly localized in the cosine phase wells. The energy bands flatten **exponentially**:
  $$\epsilon_{01} \propto \exp\left( -\sqrt{8 E_J / E_C} \right)$$
- At $E_J / E_C = 50$, the charge dispersion drops to $\epsilon_{01} \approx 9.9\ \mathrm{kHz}$ ($< 10^{-4} E_C$), rendering the qubit essentially immune to charge fluctuations ($T_2^* > 100\ \mu\mathrm{s}$).
- Simultaneously, the negative anharmonicity $\alpha \approx -E_C = -250\ \mathrm{MHz}$ decreases only as a gentle power law $\mathcal{O}((E_C/E_J)^{1/4})$. This allows selective control of the $|0\rangle \leftrightarrow |1\rangle$ transition using standard $20\ \mathrm{ns}$ shaped microwave pulses without exciting the $|2\rangle$ state.

---

### 2. Dispersive QND Readout Physics

Panel (c) maps the dispersive interaction between the qubit and microwave cavity:
- In the dispersive regime ($|\Delta| \gg g$), virtual photon exchange dresses the cavity resonance by $\pm\chi$:
  $$\omega_r(|g\rangle) = \omega_r - \chi, \quad \omega_r(|e\rangle) = \omega_r + \chi$$
- The total frequency separation between ground and excited readout peaks is:
  $$\Delta\omega_{\mathrm{res}} = 2\chi \approx 10\ \mathrm{MHz}$$
- When a probe tone is launched at $\omega_r$, the transmitted microwave field undergoes a state-dependent phase shift $\pm\phi$. Amplified by near-quantum-limited Josephson Parametric Amplifiers (JPAs) or Traveling Wave Parametric Amplifiers (TWPAs), this yields high-fidelity single-shot readout ($> 99\%$) in less than $300\ \mathrm{ns}$.
- Because $[\hat{H}_{\mathrm{disp}}, \hat{\sigma}_z] = 0$, the measurement is strictly **Quantum Non-Demolition (QND)**: it collapses the qubit into an eigenstate of $\hat{\sigma}_z$ without inducing spurious excitation or relaxation.

---

### 3. Resolving Photons One by One

Panel (d) demonstrates the reverse manifestation of the dispersive interaction: the cavity photon field dresses the qubit:
$$\omega_q(n) = \omega_{01} - 2n\chi$$
- Each individual microwave photon stored in the resonator shifts the qubit transition frequency by exactly $2\chi = 8\ \mathrm{MHz}$.
- Because this shift is much larger than the qubit absorption linewidth ($\gamma_q = 0.8\ \mathrm{MHz}$), the spectroscopy signal splits into a distinct ladder of resolved peaks for $n = 0, 1, 2, 3, \dots$.
- The peak heights directly sample the photon number statistics $P(n)$, enabling direct calibration of cavity photon occupancy and experimental verification of non-classical Fock and Schrödinger cat states in microwave cavities.


In [ ]:
# Cell 7: Interactive Circuit QED Diagnostic Suite
def analyze_transmon_cqed_system(
    ratio_EJ_EC: float = 50.0, 
    charging_energy_EC_GHz: float = 0.25, 
    coupling_g_MHz: float = 70.0, 
    cavity_freq_GHz: float = 7.0
) -> Dict[str, float]:
    '''
    Evaluates transmon frequencies, charge dispersion, anharmonicity,
    dispersive shift chi, and QND readout separation.
    '''
    EC = charging_energy_EC_GHz
    EJ = ratio_EJ_EC * EC
    
    # Diagonalize transmon
    evals_0, _ = CircuitQEDEngine.diagonalize_cooper_pair_box(EJ, EC, ng=0.0)
    omega_01 = float(evals_0[1] - evals_0[0])
    omega_12 = float(evals_0[2] - evals_0[1])
    anharm = float(omega_12 - omega_01)
    
    # Charge dispersion
    _, _, eps_01 = CircuitQEDEngine.charge_dispersion(EJ, EC)
    
    # Asymptotic formula comparison
    om_asympt, anh_asympt = CircuitQEDEngine.transmon_asymptotic_formulas(EJ, EC)
    
    # Dispersive shift calculation
    delta_detuning = omega_01 - cavity_freq_GHz # GHz
    g_GHz = coupling_g_MHz * 1e-3
    chi_GHz = - (g_GHz**2 / delta_detuning) * (EC / (delta_detuning - EC))
    chi_MHz = chi_GHz * 1e3
    
    print("=" * 68)
    print("CIRCUIT QED & TRANSMON DIAGNOSTIC REPORT")
    print("=" * 68)
    print(f"Parameters: EJ/EC = {ratio_EJ_EC:.1f} | EC = {EC:.3f} GHz | EJ = {EJ:.3f} GHz")
    print(f"- Transition Frequency:        omega_01 = {omega_01:.4f} GHz (Asymptotic: {om_asympt:.4f} GHz)")
    print(f"- Anharmonicity:               alpha = {anharm*1e3:+.1f} MHz (Expected: {-EC*1e3:+.1f} MHz)")
    print(f"- Gate Charge Dispersion:      eps_01 = {eps_01*1e6:.2f} kHz (Exp. Suppressed)")
    print(f"- Cavity Detuning:             Delta = {delta_detuning*1e3:+.1f} MHz")
    print(f"- Dispersive Shift:            chi = {chi_MHz:+.2f} MHz (Peak Separation 2*chi = {2*chi_MHz:+.2f} MHz)")
    print(f"- Dispersive Criterion:       |Delta/g| = {abs(delta_detuning/g_GHz):.1f} >> 1 (Valid)")
    print("=" * 68)
    
    return {
        "omega_01_GHz": omega_01,
        "anharmonicity_MHz": anharm * 1e3,
        "charge_dispersion_kHz": eps_01 * 1e6,
        "dispersive_shift_chi_MHz": chi_MHz,
        "readout_separation_2chi_MHz": 2.0 * chi_MHz,
    }

# Execute diagnostic
diag_results = analyze_transmon_cqed_system(
    ratio_EJ_EC=50.0, 
    charging_energy_EC_GHz=0.25, 
    coupling_g_MHz=70.0, 
    cavity_freq_GHz=7.0
)


---

## Guided Exercises & Circuit QED Research Challenges

### Challenge 1: The Schrieffer-Wolff Dispersive Transformation with Multi-Level Transmons
For a two-level qubit, the dispersive shift is simply $\chi = g^2 / \Delta$.
1. Using second-order perturbation theory on the three-level transmon Hamiltonian:
   $$\hat{H} = \hbar \omega_r \hat{a}^\dagger\hat{a} + E_0 |0\rangle\langle 0| + E_1 |1\rangle\langle 1| + E_2 |2\rangle\langle 2| + \hbar g \left( |0\rangle\langle 1|\hat{a}^\dagger + \sqrt{2}|1\rangle\langle 2|\hat{a}^\dagger + \mathrm{h.c.} \right)$$
   prove that the effective cavity frequency shift is:
   $$\chi = \chi_{01} - \frac{\chi_{12}}{2} = \frac{g^2}{\Delta} - \frac{g^2}{\Delta + \alpha} \approx -\frac{g^2}{\Delta} \frac{\alpha}{\Delta + \alpha}$$
2. Show that when $\Delta = -\alpha = E_C$, the dispersive shift exhibits a resonance pole (straddling regime).

---

### Challenge 2: Measurement-Induced Dephasing & Critical Photon Number
In dispersive readout, photon shot noise in the readout cavity induces dephasing on the transmon:
$$\Gamma_\phi = \frac{8\chi^2}{\kappa} \bar{n}_{\mathrm{cav}}$$
1. Show that when $\chi = \kappa / 2$, measurement-induced dephasing matches the measurement rate, reaching the quantum-limited measurement efficiency.
2. The dispersive approximation breaks down at the **critical photon number**:
   $$n_{\mathrm{crit}} = \frac{\Delta^2}{4g^2}$$
   Calculate $n_{\mathrm{crit}}$ for typical transmon parameters ($g = 70\ \mathrm{MHz}, \Delta = 1.5\ \mathrm{GHz}$) and explain why exceeding $n_{\mathrm{crit}}$ causes qubit state transitions (dressed dephasing).

---

### Challenge 3: The Fluxonium Qubit & Ultra-High Anharmonicity
To overcome the weak anharmonicity of the transmon while preserving noise immunity, Manucharyan et al. (2009) introduced the **Fluxonium Qubit**, shunting the junction with a superinductor ($E_L \ll E_C, E_J$):
$$\hat{H}_{\mathrm{flux}} = 4 E_C \hat{n}^2 - E_J \cos\hat{\phi} + \frac{1}{2} E_L \hat{\phi}^2$$
1. Compare the potential energy landscape of the fluxonium with the transmon.
2. Explain how fluxonium achieves both a sweet spot protected against flux noise and an enormous positive anharmonicity ($\alpha > 1\ \mathrm{GHz}$).


In [ ]:
# Cell 9: Automated Benchmark & Verification Suite
def run_benchmark_suite():
    print("=" * 70)
    print("STARTING CIRCUIT QED & TRANSMON BENCHMARK SUITE")
    print("=" * 70)
    
    EC = 0.25 # GHz
    EJ = 50.0 * EC # EJ/EC = 50

    # -----------------------------------------------------------------
    # Test 1: Transmon Transition Frequency Asymptotic Match (< 1%)
    # -----------------------------------------------------------------
    print("Test 1: Verifying Transmon Transition Frequency Asymptotic Formula...")
    evals, _ = CircuitQEDEngine.diagonalize_cooper_pair_box(EJ, EC, ng=0.0)
    omega_01_num = evals[1] - evals[0]
    omega_01_asympt, _ = CircuitQEDEngine.transmon_asymptotic_formulas(EJ, EC)
    
    rel_err_omega = abs(omega_01_num - omega_01_asympt) / omega_01_num
    assert rel_err_omega < 0.01, f"Transition frequency diverged from asymptotic formula: {omega_01_num} vs {omega_01_asympt}"
    print(f"  -> Numerical omega_01:   {omega_01_num:.6f} GHz")
    print(f"  -> Asymptotic formula:  {omega_01_asympt:.6f} GHz (Relative Error: {rel_err_omega:.2e} < 1%) (PASS)")

    # -----------------------------------------------------------------
    # Test 2: Negative Anharmonicity Match (alpha ~ -EC)
    # -----------------------------------------------------------------
    print("\nTest 2: Verifying Negative Anharmonicity alpha ~ -EC...")
    omega_12_num = evals[2] - evals[1]
    anharm_num = omega_12_num - omega_01_num
    expected_anharm = -EC
    
    # At EJ/EC = 50, higher-order corrections deviate by ~15% from pure -EC
    rel_diff_anh = abs(anharm_num - expected_anharm) / abs(expected_anharm)
    assert anharm_num < 0.0, f"Anharmonicity was positive: {anharm_num}"
    assert rel_diff_anh < 0.20, f"Anharmonicity diverged: {anharm_num} vs {expected_anharm}"
    print(f"  -> Numerical anharmonicity: alpha = {anharm_num*1e3:.2f} MHz")
    print(f"  -> Leading-order target:    -EC   = {expected_anharm*1e3:.2f} MHz (Match within 15%) (PASS)")

    # -----------------------------------------------------------------
    # Test 3: Exponential Suppression of Charge Dispersion (< 10^-4 * EC)
    # -----------------------------------------------------------------
    print("\nTest 3: Verifying Exponential Charge Dispersion Suppression...")
    _, _, eps_01 = CircuitQEDEngine.charge_dispersion(EJ, EC)
    threshold_disp = 1e-4 * EC # 25 kHz
    assert eps_01 < threshold_disp, f"Charge dispersion was insufficiently suppressed: {eps_01} >= {threshold_disp}"
    print(f"  -> Charge dispersion at EJ/EC=50: eps_01 = {eps_01*1e6:.2f} kHz < {threshold_disp*1e6:.2f} kHz (PASS)")

    # -----------------------------------------------------------------
    # Test 4: Dispersive Cavity Shift Separation (Delta_omega = 2*chi)
    # -----------------------------------------------------------------
    print("\nTest 4: Verifying Dispersive Cavity Shift Peak Separation 2*chi...")
    omega_r = 7.0
    chi = 0.005 # 5 MHz
    kappa = 0.002 # 2 MHz
    f_test = np.linspace(6.99, 7.01, 1000)
    
    _, T_g = CircuitQEDEngine.dispersive_cavity_transmission(f_test, omega_r, chi, kappa, qubit_state="g")
    _, T_e = CircuitQEDEngine.dispersive_cavity_transmission(f_test, omega_r, chi, kappa, qubit_state="e")
    
    peak_g = f_test[np.argmin(T_g)]
    peak_e = f_test[np.argmin(T_e)]
    measured_2chi = peak_e - peak_g
    expected_2chi = 2.0 * chi
    
    assert np.isclose(measured_2chi, expected_2chi, atol=1e-4), f"Dispersive peak separation mismatch: {measured_2chi} vs {expected_2chi}"
    print(f"  -> Measured Readout Peak Separation: {measured_2chi*1e3:.2f} MHz == 2*chi = {expected_2chi*1e3:.2f} MHz (PASS)")

    # -----------------------------------------------------------------
    # Test 5: Photon-Number-Resolved Spectroscopy Peak Ladder
    # -----------------------------------------------------------------
    print("\nTest 5: Verifying Photon-Number Splitting Ladder (omega_q - 2*n*chi)...")
    omega_q_test = 5000.0 # MHz
    chi_test = 4.0        # MHz
    gamma_q_test = 0.5    # MHz
    # Test with n=2 Fock state (single peak at omega_q - 4*chi = 4984 MHz)
    fock_2_dist = np.array([0.0, 0.0, 1.0, 0.0])
    w_scan = np.linspace(4970.0, 5010.0, 1000)
    spec_fock = CircuitQEDEngine.photon_number_resolved_spectrum(w_scan, omega_q_test, chi_test, gamma_q_test, fock_2_dist)
    
    peak_fock_loc = w_scan[np.argmax(spec_fock)]
    expected_loc = omega_q_test - 2.0 * 2.0 * chi_test # 4984 MHz
    assert np.isclose(peak_fock_loc, expected_loc, atol=0.1), f"Photon splitting peak location mismatch: {peak_fock_loc} vs {expected_loc}"
    print(f"  -> Qubit peak for n=2 photons: {peak_fock_loc:.2f} MHz == omega_q - 4*chi = {expected_loc:.2f} MHz (PASS)")

    print("=" * 70)
    print("ALL 5 CIRCUIT QED BENCHMARK TESTS PASSED WITH ZERO ERRORS!")
    print("=" * 70)

run_benchmark_suite()


---

## Summary & Pedagogical Bridge to Notebook 6.5

### Key Takeaways from Notebook 6.4:
1. **The Transmon Advantage**: By operating at $E_J / E_C \gg 1$, charge dispersion is suppressed exponentially ($\epsilon \propto e^{-\sqrt{8E_J/E_C}}$), eliminating $1/f$ charge noise dephasing while maintaining adequate negative Duffing anharmonicity ($\alpha \approx -E_C$) for selective two-level quantum control.
2. **Dispersive Hamiltonian**: In the far-detuned regime ($|\Delta| \gg g$), virtual photon exchange shifts the cavity frequency by $\pm\chi$ depending on the qubit state ($\hat{H}_{\mathrm{disp}} \approx \hbar(\omega_r + \chi \hat{\sigma}_z)\hat{a}^\dagger\hat{a}$).
3. **QND Readout**: Because $[\hat{H}_{\mathrm{disp}}, \hat{\sigma}_z] = 0$, measuring the transmission of a microwave probe tone through the cavity provides fast, single-shot, non-destructive measurement of the qubit.
4. **Photon-Number-Resolved Spectroscopy**: In the strong dispersive regime ($\chi > \kappa, \gamma$), the qubit transition splits into discrete peaks $\omega_q - 2n\chi$, allowing direct counting of single microwave photons and reconstruction of non-classical cavity fields.

---

### Looking Ahead to Notebook 6.5: Bosonic Quantum Error Correction (GKP & Cat Codes)
Circuit QED provides not only superconducting qubits, but also long-lived 3D microwave cavities ($T_1 \sim 1\ \mathrm{ms}$) with infinite-dimensional Hilbert spaces. In **Notebook 6.5**, we explore **Hardware-Efficient Bosonic Quantum Error Correction**:
- Storing and protecting quantum information in harmonic oscillator states rather than physical multi-qubit arrays.
- **Gottesman-Kitaev-Preskill (GKP) Grid States**: Periodic grid superpositions in phase space stabilized by modular displacement operators $\hat{S}_x = e^{i 2\sqrt{\pi} \hat{p}}$ and $\hat{S}_p = e^{-i 2\sqrt{\pi} \hat{x}}$.
- **Schrödinger Cat Codes**: Protecting logical qubits against single-photon loss using 4-component coherent state superpositions.
- **Autonomous Error Correction**: Engineering non-linear dissipation (two-photon loss $\kappa_2 \mathcal{D}[\hat{a}^2 - \alpha^2]$) via cQED ancillas to continuously stabilize the code space.
